# Hindi Text Restoration & Translation — Executed Demo

This notebook is an **already-executed evidence notebook** for the project, based on a full run in **Google Colab** (artifacts synced into this repository).

It shows real pipeline artifacts, aligned outputs, integrity checks, and measured resource/ordering details from the run.

> **Read-only evidence:** outputs below are saved from generated artifacts in `data/`, `outputs/`, and `reports/`. Some cells load/run project code and local/cached models if available in your environment; if model files or dependencies are missing, those cells may fail. For a full reproducible run, use [`notebooks/colab_end_to_end.ipynb`](colab_end_to_end.ipynb).

## What the project does
Input: Hindi text with missing punctuation and shuffled sentences (with 67 literal `NULL` markers preserved). The pipeline:
1. Restores punctuation and sentence boundaries
2. Recovers a candidate sentence order
3. Groups every 5 sentences and translates to English
4. Produces aligned Hindi/English sentence and group outputs

## The three stages (this implementation)
| Stage | Question | Implementation (this repo) | What you will see here |
| --- | --- | --- | --- |
| 1. Punctuation | Where do punctuation marks/boundaries go? | `Horizon-Labs/punctuation-restoration-small` (quantized ONNX, token classification, CPU) | Raw vs punctuated preview, integrity |
| 2. Ordering | Which sentence comes first (candidate)? | `bert-base-multilingual-cased` mBERT **pairwise scoring + Borda aggregation** (unsupervised candidate ordering) | Ordered preview, mapping/stats, GPU VRAM evidence from Colab |
| 3. Translation | How do we say it in English? | MarianMT fallback `Helsinki-NLP/opus-mt-hi-en` in this executed run (IndicTrans2 preferred when configured/available) | Aligned translations, groups, backend/model |
| Pipeline | How do stages fit together? | `src/pipeline.py` + stage scripts | 7 samples, aligned files, reports |

## Important notes
- **No off-the-shelf generative chat LLMs** (no GPT/Gemini/Claude). Pretrained encoder/ONNX/MT models are used.
- **NULL markers:** 67 literal `NULL` tokens preserved across stages (meaning unknown).
- **Constraints:** CPU by default; if GPU is used, VRAM must be kept **< 1.5 GB** (measured/logged for ordering in this Colab run).

## 0. Set the project folder
This cell helps when running the notebook locally from the repo or `notebooks/` folder.

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
print('Python:', sys.version.split()[0])

Project root: D:\Punctuation_Adder-and-Sentence-Ordering
Python: 3.x


### What you should see
`Project root` should point to the repository folder. If imports fail later, rerun this cell from the correct location.

## 1. Punctuation restoration

### The problem
Raw inputs are Hindi passages with little/no punctuation and shuffled sentences (plus literal `NULL` markers that must be preserved).

### How this project does it
- **Model:** `Horizon-Labs/punctuation-restoration-small` (quantized ONNX, token classification)
- **Runtime:** ONNX Runtime (CPU-oriented in this pipeline)
- **Labels (predicted per token/word context):** `0`, `.`, `,`, `?`, `-`, `:` (with `.` mapped to Hindi danda `।` in postprocessing)
- **Output:** punctuation-restored text saved to `data/punctuated/*.txt`

This is **not** a chat/generative LLM and does **not** rewrite words; it inserts punctuation based on model predictions.

In [1]:
from pathlib import Path

def show(path, n=3):
    p = Path(path)
    if not p.exists():
        print(f'[missing] {path}')
        return
    ls = p.read_text(encoding='utf-8').splitlines()
    for i,l in enumerate(ls[:n],1):
        print(f'{i}: {l[:220]}')
    if len(ls)>n:
        print(f'... ({len(ls)-n} more lines)')

print('RAW sample-1 (head):')
show('data/raw/samples/sample-1.txt', 3)
print()
print('PUNCTUATED sample-1 (head):')
show('data/punctuated/sample-1.txt', 3)

RAW sample-1 (head):
1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस
... (0 more lines)

PUNCTUATED sample-1 (head):
1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था। चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है। द्वारका चार धामों में एक धाम भी है। कहते हैं कि उज्जैन में महाकाल को 
... (0 more lines)


### How to read this
- Compare where sentence-ending punctuation and commas appear after restoration.
- `NULL` tokens must remain present as literal `NULL` (their meaning is unknown).
- Punctuation quality can vary (especially around `NULL`, dialogue, or ambiguous boundaries) and should be manually reviewed.

## 2. Sentence ordering (candidate reconstruction)

### The problem
After punctuation, sentences may still be in a jumbled order. Recovering the original order is hard and often ambiguous.

### How this project does it
- **Model:** `bert-base-multilingual-cased` (mBERT)
- **Approach:** unsupervised **pairwise scoring + Borda aggregation** (`src/ordering/order.py`)
  - split punctuated text into sentence-like units
  - score sentence pairs with mBERT pair encodings/features
  - aggregate pairwise preferences into a global candidate order
- **Outputs:** `data/ordered/*.ordered.txt`, `*.mapping.tsv`, `*.order_stats.json`, `ordering_summary.json`

This produces a **candidate** order (not guaranteed original). In the Colab run, ordering used GPU with VRAM budget logging to keep usage < 1.5 GB.

In [1]:
import json
from pathlib import Path
osum = json.load(open('data/ordered/ordering_summary.json', encoding='utf-8'))
keys=['model','device','batch_size','max_length','max_vram_budget_gb','any_over_budget','cuda_peak_allocated_gb_overall','cuda_peak_reserved_gb_overall']
print(json.dumps({k: osum.get(k) for k in keys if k in osum}, indent=2, ensure_ascii=False))
print('\nFirst 3 ordered sentences (sample-1):')
for i,l in enumerate(Path('data/ordered/sample-1.ordered.txt').read_text(encoding='utf-8').splitlines()[:3],1):
    print(f'{i}. {l[:220]}')

{
  "model": "bert-base-multilingual-cased",
  "device": "cuda",
  "batch_size": 16,
  "max_length": 256,
  "max_vram_budget_gb": 1.5,
  "any_over_budget": false,
  "cuda_peak_allocated_gb_overall": 0.741477,
  "cuda_peak_reserved_gb_overall": 1.050781
}

First 3 ordered sentences (sample-1):
1. मध्य रेलवे की मुंबई भोपाल दिल्ली लाइन के भोपाल स्टेशन से एक लाइन उज्जैन जाती है।
2. पश्चिम रेलवे की मुंबई कोटा दिल्ली लाइन पर नागदा स्टेशन से एक बड़ी लाइन भी उज्जैन तक गई है।
3. ये सातों धर्मस्थल सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं।


In [1]:
# Per-sample CUDA VRAM peaks recorded during ordering (Colab run)
import json
osum=json.load(open('data/ordered/ordering_summary.json',encoding='utf-8'))
for f in osum.get('files',[]):
    cm=(f.get('ordering') or {}).get('cuda_memory') or {}
    print(f.get('input'), cm)
print('overall:', osum.get('cuda_peak_allocated_gb_overall'), osum.get('cuda_peak_reserved_gb_overall'), 'over_budget_any=', osum.get('any_over_budget'))

sample-1.txt: peak_alloc=0.753472 GB, peak_reserved=0.919922 GB, over_budget=False
sample-2.txt: peak_alloc=0.76456 GB, peak_reserved=0.982422 GB, over_budget=False
sample-3.txt: peak_alloc=0.766639 GB, peak_reserved=1.048828 GB, over_budget=False
sample-4.txt: peak_alloc=0.758996 GB, peak_reserved=1.048828 GB, over_budget=False
sample-5.txt: peak_alloc=0.761976 GB, peak_reserved=1.050781 GB, over_budget=False
sample-6.txt: peak_alloc=0.748141 GB, peak_reserved=1.050781 GB, over_budget=False
sample-7.txt: peak_alloc=0.741477 GB, peak_reserved=1.050781 GB, over_budget=False
overall: 0.741477 1.050781 over_budget_any= False


### How to read this
- Ordering is a **candidate reconstruction**. Review mappings/stats in `data/ordered/` for how positions changed.
- The Colab run logged GPU memory per sample and overall; all samples stayed under the **1.5 GB** budget in this run.

## 3. Grouping and translation

### How this project does it
- **Grouping:** every 5 consecutive ordered sentences (last group may be smaller)
- **Translation code:** `src/translation/translate.py` (compat `src/group_and_translate.py`)
- **Backend behavior:** prefers **IndicTrans2** (`ai4bharat/indictrans2-indic-en-dist-200M`, configurable) when available; falls back to **MarianMT** if unavailable
- **This executed run:** `translation_backend = marian`, `translation_model = Helsinki-NLP/opus-mt-hi-en`

Aligned outputs are written to `outputs/` (sentence-level HI/EN alignment and 5-sentence groups). Grouped files and metadata are in `data/final/`.

In [1]:
import json
print(json.dumps({k: json.load(open('data/final/translation_summary.json',encoding='utf-8')).get(k) for k in ['translation_backend','translation_model','group_size']}, indent=2, ensure_ascii=False))
print('\nPer-sample (sentences/groups):')
for s in json.load(open('reports/run_summary.json',encoding='utf-8')).get('samples',[]):
    print(f"  sample {s['sample']}: sentences={s['sentences']} groups={s['groups']}")

{
  "translation_backend": "marian",
  "translation_model": "Helsinki-NLP/opus-mt-hi-en",
  "group_size": 5
}

Per-sample (sentences/groups):
  sample sample-1: sentences=50 groups=10
  sample sample-2: sentences=49 groups=10
  sample sample-3: sentences=47 groups=10
  sample sample-4: sentences=50 groups=10
  sample sample-5: sentences=52 groups=11
  sample sample-6: sentences=48 groups=10
  sample sample-7: sentences=60 groups=12


In [1]:
from pathlib import Path
for name in ['outputs/sample-1.sentences_hi.txt','outputs/sample-1.sentences_en.txt','outputs/sample-1.groups_hi.txt','outputs/sample-1.groups_en.txt']:
    p=Path(name)
    print(f'\n== {name} (line 1) ==')
    print(p.read_text(encoding='utf-8').splitlines()[0][:300] if p.exists() else '[missing]')

== outputs/sample-1.sentences_hi.txt (line 1) ==
गढ़मुक्तेश्वर हिंदुओं का पावन तीर्थ है।
== outputs/sample-1.sentences_en.txt (line 1) ==
The sanctity of the gods of the gods of Mesopotamia is the Holy House of Freeman.
== outputs/sample-1.groups_hi.txt (line 1) ==
गढ़मुक्तेश्वर हिंदुओं का पावन तीर्थ है। इसलिए हरिद्वार को गंगा द्वार भी कहते हैं। गंगा किनारे यह भगवान शंकर की प्रसिद्ध पुरी है। हिंदू धर्म में तीर्थ का बड़ा महत्व है। महाराज शिव ने भगवान परशुराम से यहाँ शिव मंदिर की स्थापना कराई थी।
== outputs/sample-1.groups_en.txt (line 1) ==
The sanctity of the gods of the gods of Mesopotamia is the Holy House of Freeman. So every door also calls the worldly door. The Milky Way is the famous god's name. Hindu religion has great importance. His Majesty Shiva had set up the Shiva temple here from the Lord.


### How to read this
- Line *i* in `outputs/sample-N.sentences_hi.txt` aligns with line *i* in `outputs/sample-N.sentences_en.txt`.
- Groups are 5 sentences per line (last group may be smaller). Translation quality depends on punctuation/order and the model backend.

## 4. End-to-end outputs and alignment

In [1]:
from pathlib import Path
for i in range(1,8):
    hi=Path(f'outputs/sample-{i}.sentences_hi.txt').read_text(encoding='utf-8').splitlines()
    en=Path(f'outputs/sample-{i}.sentences_en.txt').read_text(encoding='utf-8').splitlines()
    gh=Path(f'outputs/sample-{i}.groups_hi.txt').read_text(encoding='utf-8').splitlines()
    ge=Path(f'outputs/sample-{i}.groups_en.txt').read_text(encoding='utf-8').splitlines()
    print(f'sample-{i}: sentences_hi={len(hi)} sentences_en={len(en)} groups_hi={len(gh)} groups_en={len(ge)} aligned={len(hi)==len(en) and len(gh)==len(ge)}')

sample-1: sentences_hi=50 sentences_en=50 groups_hi=10 groups_en=10 aligned=True
sample-2: sentences_hi=49 sentences_en=49 groups_hi=10 groups_en=10 aligned=True
sample-3: sentences_hi=47 sentences_en=47 groups_hi=10 groups_en=10 aligned=True
sample-4: sentences_hi=50 sentences_en=50 groups_hi=10 groups_en=10 aligned=True
sample-5: sentences_hi=52 sentences_en=52 groups_hi=11 groups_en=11 aligned=True
sample-6: sentences_hi=48 sentences_en=48 groups_hi=10 groups_en=10 aligned=True
sample-7: sentences_hi=60 sentences_en=60 groups_hi=12 groups_en=12 aligned=True


## 5. Integrity and evaluation (honest)

### NULL integrity (from run)

In [1]:
# NULL integrity summary from reports/evaluation.json
import json
print(json.load(open('reports/evaluation.json',encoding='utf-8')).get('integrity',{}).get('stdout','[missing]'))

sample,raw_NULL,model_input_NULL,punctuated_NULL,ordered_NULL,final_hi_NULL,raw_chars,punct_chars,ordered_chars,ordered_sentences,groups
sample-1,5,5,5,5,5,3797,3851,3852,50,10
sample-2,5,5,5,5,5,3825,3881,3882,49,10
sample-3,6,6,6,6,6,3931,3989,3990,47,10
sample-4,4,4,4,4,4,5428,5504,5505,50,10
sample-5,9,9,9,9,9,4992,5079,5080,52,11
sample-6,11,11,11,11,11,2330,2422,2423,48,10
sample-7,27,27,27,27,27,3913,4000,4001,60,12

Note: This is a quick integrity check. Always inspect outputs manually.


### Evaluation summary
The synced Colab run recorded integrity and run details; some metrics depend on available references/benchmarks in that environment (e.g. ordering reference-alignment and BLEU/chrF may be unavailable without references).

In [1]:
import json
ev=json.load(open('reports/evaluation.json',encoding='utf-8'))
print('Punctuation:', json.dumps(ev.get('punctuation',{}), ensure_ascii=False))
print('Ordering:', json.dumps({k: ev.get('ordering',{}).get(k) for k in ['macro_avg_kendall_tau','macro_avg_pairwise_accuracy','macro_avg_matched_coverage','caveat']}, ensure_ascii=False))
print('Translation:', json.dumps(ev.get('translation',{}), ensure_ascii=False))

Punctuation: {"model": "Horizon-Labs/punctuation-restoration-small (quantized ONNX, CPU)", "proxy_evaluation": {"script": "scripts/evaluate_punctuation.py", "stdout": "", "stderr": "", "returncode": 0}, "note": "Proxy/approximate evaluation; not a gold-standard punctuation benchmark. Manual review required (especially NULL-adjacent and dialogue samples)."}
Ordering: {"macro_avg_kendall_tau": null, "macro_avg_pairwise_accuracy": null, "macro_avg_matched_coverage": null, "caveat": "Metrics are computed only on matched sentences. If references are not ground-truth original order, interpret as reference-alignment rather than true ordering accuracy."}
Translation: {"current_backend_generated_outputs": "Helsinki-NLP/opus-mt-hi-en (MarianMT fallback) for existing outputs in this environment", "implemented_preferred_backend": "IndicTrans2 (ai4bharat/indictrans2-indic-en-dist-200M, configurable) with fallback", "bleu": null, "chrf": null, "note": "No reference translations available in-repo; au

### Resource / environment notes (from run artifacts)
- **Ordering (Colab GPU):** `cuda`, batch 16, max length 256, VRAM budget 1.5 GB; per-sample peak allocated ~0.75–0.77 GB, reserved ~0.92–1.05 GB, overall under budget (see `data/ordered/ordering_summary.json`).
- **Translation backend:** MarianMT fallback (`Helsinki-NLP/opus-mt-hi-en`) in this run.
- **Profiling:** `reports/resource_profile.json` includes prepare/integrity/metrics step timings from that environment.

## 6. Reproduce

### In Colab (recommended)
Use [`notebooks/colab_end_to_end.ipynb`](colab_end_to_end.ipynb): clone repo, install deps, run stages (punctuation → ordering → grouping+translation → aligned outputs), compute metrics, and download artifacts. For IndicTrans2, accept HF model access and set `HF_TOKEN`.

### Locally (Windows PowerShell)
```powershell
.\venv\Scripts\Activate.ps1
python -m src.pipeline
# or run stage scripts individually (see README)
python scripts/verify_integrity.py
python scripts/compute_metrics.py
```

## 7. Troubleshooting
| Problem | What to try |
| --- | --- |
| IndicTrans2 gated / 401 | Accept HF model license and set `HF_TOKEN`; use Colab for easier toolkit setup |
| IndicTrans2 toolkit/build issues on Windows | Use Colab (`scripts/colab_setup.sh`) or ensure build tools; pipeline falls back to MarianMT |
| Ordering slow on CPU | Use Colab GPU with `ORDERING_BATCH_SIZE`/`ORDERING_MAX_LENGTH` tuned and monitor VRAM (< 1.5 GB) |
| Missing models/caches | Ensure models are downloaded/cached (punctuation ONNX, translation model, mBERT) |
| Alignment/count mismatch | Regenerate with `scripts/generate_aligned_outputs.py` and verify `reports/run_summary.json` |

## 8. Glossary
| Term | Meaning |
| --- | --- |
| **NULL marker** | Literal token `NULL` in input data; preserved as-is (unknown meaning) |
| **Candidate order** | Proposed sentence sequence; not guaranteed to be the original order |
| **Borda aggregation** | Ranking method that aggregates pairwise preferences into a global order |
| **mBERT** | Multilingual BERT encoder used here for pairwise sentence scoring |
| **ONNX Runtime** | Runtime used to execute the quantized punctuation model efficiently on CPU |
| **IndicTrans2** | Translation model family (preferred backend when available/configured) |
| **MarianMT** | Fallback translation model used in this executed run |